# Data Center Energy Efficiency — AI-driven cooling control
**Project 19** · Datasets: *Server Workload Traces* + *Data Center Temperature & Humidity Sensors* (Kaggle)

| Module | Technique | Role |
|---|---|---|
| Forecast | **LSTM** | Predict the CPU load of the servers (next 30 min) |
| Thermal analysis | **CNN** | Read 2D rack heat maps → probability of a hotspot |
| Control | **Random Forest** | Choose the HVAC supply-air setpoint dynamically |
| Evaluation | Energy model | kWh, cost and CO₂ saved vs. a fixed setpoint |

**How to run on Kaggle:** *Add Data* → attach both datasets → turn on a GPU (T4) → *Run All*.
If the CSV files are not found, the notebook falls back to a **synthetic demo dataset**, so it always runs end to end.

## 1. Setup & configuration

### Installation des dépendances

Depuis la racine du dépôt, ex?cutez `python -m pip install -r requirements-notebook.txt` avant de lancer les cellules.

In [ ]:
import os, glob, math, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
tf.random.set_seed(SEED)

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             roc_auc_score, f1_score, confusion_matrix)

# ======================== CONFIG ========================
WORKLOAD_CSV  = None    # e.g. "/kaggle/input/<dataset>/<file>.csv"  (None = auto-detect)
SENSOR_CSV    = None    # same for the temperature / humidity dataset
MAX_ROWS      = 2_000_000   # safety limit when reading huge CSV files
STEP          = "5min"  # common time step for all series
LOOKBACK      = 36      # LSTM history  : 36 steps = 3 h
HORIZON       = 6       # forecast span : 6 steps  = 30 min
MAX_SENSORS   = 64      # max sensors used for the thermal map (8x8)
HOT_QUANTILE  = 0.85    # hotspot = max rack temperature above this quantile (train set)
HOT_THRESHOLD = None    # or a fixed value in deg C, e.g. 27.0 (ASHRAE upper limit)

# ---- Energy model assumptions (state them in the report) ----
N_SERVERS       = 200      # servers in the room
P_IDLE_KW       = 0.10     # power of one idle server (kW)
P_MAX_KW        = 0.30     # power of one fully loaded server (kW)
BASE_SETPOINT   = 20.0     # baseline: fixed supply-air setpoint (deg C)
SP_MIN, SP_MAX  = 16.0, 26.0   # allowed setpoint range
K_PASS          = 0.6      # deg C change of rack inlet temp per deg C change of setpoint
SAFETY_MARGIN   = 0.5      # deg C kept below the hot threshold by the controller
CO2_KG_PER_KWH  = 0.45     # grid emission factor -> use your local value
PRICE_PER_KWH   = 0.10     # electricity price (any currency)

OUT = "/kaggle/working" if os.path.exists("/kaggle/working") else "./outputs"
os.makedirs(OUT, exist_ok=True)
print("TensorFlow", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))

: 

## 2. Data loading (auto-detection of the Kaggle files)

In [ ]:
csvs = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
for p in csvs:
    print(f"{os.path.getsize(p)/1e6:8.1f} MB  {p}")

def pick(patterns, exclude=()):
    for p in csvs:
        n = p.lower()
        if any(k in n for k in patterns) and not any(k in n for k in exclude):
            return p
    return None

if WORKLOAD_CSV is None:
    WORKLOAD_CSV = pick(["workload", "trace", "cpu", "usage", "server"],
                        exclude=["temp", "humid", "sensor"])
if SENSOR_CSV is None:
    SENSOR_CSV = pick(["temp", "humid", "sensor", "thermal"])
print("\nWorkload file:", WORKLOAD_CSV)
print("Sensor file  :", SENSOR_CSV)

In [ ]:
def find_time_col(df):
    """Return (column_name, parsed_datetime_series) for the first usable time column."""
    keys = ["time", "date", "stamp", "ts"]
    named = [c for c in df.columns if any(k in c.lower() for k in keys)]
    for c in named + [c for c in df.columns if c not in named]:
        s = df[c]
        try:
            if pd.api.types.is_numeric_dtype(s):
                if c in named and s.nunique() > 10:
                    v = float(s.dropna().iloc[0])
                    unit = "ms" if v > 1e11 else "s"
                    return c, pd.to_datetime(s, unit=unit, errors="coerce")
            else:
                t = pd.to_datetime(s, errors="coerce")
                if t.notna().mean() > 0.9:
                    return c, t
        except Exception:
            continue
    return None, None


def load_workload(path):
    """-> pd.Series 'cpu' in [0,1], mean over servers, resampled to STEP."""
    df = pd.read_csv(path, nrows=MAX_ROWS)
    tcol, t = find_time_col(df)
    if tcol is None:
        raise ValueError(f"No time column in workload file. Columns: {list(df.columns)}")
    df["_t"] = t
    df = df.dropna(subset=["_t"])
    num = [c for c in df.select_dtypes("number").columns if c != tcol]
    cpu_cols = [c for c in num if "cpu" in c.lower()] or num[:1]
    prefer = [c for c in cpu_cols if any(k in c.lower() for k in ["util", "usage", "avg", "load", "mean"])]
    col = (prefer or cpu_cols)[0]
    print("Workload: using CPU column ->", col)
    s = pd.Series(df[col].astype(float).values, index=pd.DatetimeIndex(df["_t"].values))
    s = s.groupby(level=0).mean().sort_index()
    if s.max() > 1.5:
        s = s / (100.0 if s.max() <= 100 else s.max())
    s = s.clip(0, 1).resample(STEP).mean().interpolate(limit_direction="both")
    return s.rename("cpu")


def load_sensors(path):
    """-> (temps DataFrame [time x sensors], humidity Series), resampled to STEP."""
    df = pd.read_csv(path, nrows=MAX_ROWS)
    tcol, t = find_time_col(df)
    if tcol is None:
        raise ValueError(f"No time column in sensor file. Columns: {list(df.columns)}")
    df["_t"] = t
    df = df.dropna(subset=["_t"])
    df["_t"] = df["_t"].dt.floor(STEP)
    num = [c for c in df.select_dtypes("number").columns if c != tcol]
    temp_cols = [c for c in num if "temp" in c.lower()]
    hum_cols = [c for c in num if "hum" in c.lower()]
    if not temp_cols:
        temp_cols = [c for c in num if c not in hum_cols]
    id_keys = ["sensor", "rack", "device", "node", "location", "zone", "id"]
    id_cols = [c for c in df.columns
               if any(k in c.lower() for k in id_keys) and c not in temp_cols + hum_cols + ["_t", tcol]
               and 2 <= df[c].nunique() <= 500]
    if id_cols and len(temp_cols) >= 1:                      # long format -> pivot
        idc = id_cols[0]
        print("Sensors: long format, id column ->", idc, "| temp column ->", temp_cols[0])
        temps = df.pivot_table(index="_t", columns=idc, values=temp_cols[0], aggfunc="mean")
        hum = (df.pivot_table(index="_t", columns=idc, values=hum_cols[0], aggfunc="mean").mean(axis=1)
               if hum_cols else None)
    else:                                                     # wide format
        print("Sensors: wide format, temperature columns ->", temp_cols[:8], "...")
        g = df.groupby("_t")
        temps = g[temp_cols].mean()
        hum = g[hum_cols].mean().mean(axis=1) if hum_cols else None
    temps = temps.sort_index().resample(STEP).mean().interpolate(limit_direction="both")
    if temps.stack().median() > 60:                           # looks like Fahrenheit
        temps = (temps - 32) * 5.0 / 9.0
        print("Temperatures converted from F to C")
    keep = temps.var().sort_values(ascending=False).index[:MAX_SENSORS]
    temps = temps[[c for c in temps.columns if c in keep]]
    temps.columns = [f"T_{i}" for i in range(temps.shape[1])]
    if hum is None:
        hum = pd.Series(45.0, index=temps.index)
    hum = hum.sort_index().resample(STEP).mean().interpolate(limit_direction="both")
    return temps, hum.rename("hum")


def make_synthetic(days=21, n_side=6):
    """Fallback demo data: daily load pattern + a fixed hot zone in a 6x6 rack room."""
    idx = pd.date_range("2024-01-01", periods=days * 288, freq="5min")
    h = idx.hour + idx.minute / 60
    base = 0.45 + 0.25 * np.sin(2 * np.pi * (h - 9) / 24) + 0.06 * np.sin(2 * np.pi * idx.dayofweek / 7)
    spikes = (np.random.rand(len(idx)) < 0.01) * np.random.uniform(0.15, 0.35, len(idx))
    cpu = pd.Series(np.clip(base + spikes + np.random.normal(0, 0.04, len(idx)), 0.02, 1.0), index=idx)
    cpu = cpu.rolling(3, min_periods=1).mean()
    ys, xs = np.mgrid[0:n_side, 0:n_side]
    heat = np.exp(-(((xs - 4) ** 2 + (ys - 1) ** 2) / 6.0)).ravel()
    lagged = cpu.shift(3).bfill().values
    temps = pd.DataFrame({f"T_{k}": 19 + 6 * lagged + 4 * heat[k] + np.random.normal(0, 0.3, len(idx))
                          for k in range(n_side * n_side)}, index=idx)
    hum = pd.Series(45 - 8 * lagged + np.random.normal(0, 1, len(idx)), index=idx, name="hum")
    return cpu.rename("cpu"), temps, hum

In [ ]:
USING_SYNTHETIC = not (WORKLOAD_CSV and SENSOR_CSV)
if USING_SYNTHETIC:
    print("No compatible CSV files found; using SYNTHETIC demo data")
    cpu, temps, hum = make_synthetic()
else:
    cpu = load_workload(WORKLOAD_CSV)
    temps, hum = load_sensors(SENSOR_CSV)

# --- Time alignment: the two datasets come from different sources -----------------
# If their time ranges do not overlap, shift the workload series onto the sensor timeline.
# (Assumption to document in the report.)
start, end = max(cpu.index[0], temps.index[0]), min(cpu.index[-1], temps.index[-1])
span = min(cpu.index[-1] - cpu.index[0], temps.index[-1] - temps.index[0])
overlap = (end - start) / span if span > pd.Timedelta(0) else 0
if overlap < 0.5:
    shift = temps.index[0] - cpu.index[0]
    cpu.index = cpu.index + shift
    print(f"Time ranges barely overlap -> workload shifted by {shift}")

data = temps.join(cpu, how="inner").join(hum, how="left")
data["hum"] = data["hum"].interpolate(limit_direction="both").fillna(45.0)
data = data.dropna()
sensor_cols = [c for c in data.columns if c.startswith("T_")]
data["temp_mean"] = data[sensor_cols].mean(axis=1)
data["temp_max"] = data[sensor_cols].max(axis=1)
hod = data.index.hour + data.index.minute / 60
data["hour_sin"] = np.sin(2 * np.pi * hod / 24)
data["hour_cos"] = np.cos(2 * np.pi * hod / 24)

N = len(data)
i_tr, i_va = int(0.70 * N), int(0.85 * N)          # chronological split 70/15/15
print(f"Aligned dataset: {N} steps of {STEP}, {len(sensor_cols)} sensors "
      f"({data.index[0]} -> {data.index[-1]})")
print(f"Train/Val/Test = {i_tr}/{i_va - i_tr}/{N - i_va}")
data.head()

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
d = data.iloc[: 288 * 3]
ax[0].plot(d.index, d["cpu"]);         ax[0].set_ylabel("CPU load (0-1)")
ax[1].plot(d.index, d["temp_mean"], label="mean"); ax[1].plot(d.index, d["temp_max"], label="max")
ax[1].set_ylabel("Rack temp (°C)"); ax[1].legend()
ax[2].plot(d.index, d["hum"]);         ax[2].set_ylabel("Humidity (%)")
plt.suptitle("First 3 days of aligned data"); plt.tight_layout(); plt.show()
print("Correlation CPU load vs mean temperature:", round(data["cpu"].corr(data["temp_mean"]), 3))

## 3. LSTM — CPU load forecasting

In [ ]:
feat = ["cpu", "hour_sin", "hour_cos", "temp_mean"]
cpu_idx = feat.index("cpu")
scaler = MinMaxScaler().fit(data[feat].iloc[:i_tr])
X_all = scaler.transform(data[feat]).astype("float32")
c_min, c_rng = scaler.data_min_[cpu_idx], scaler.data_range_[cpu_idx]
inv_cpu = lambda a: a * c_rng + c_min

# Decision time t: input = [t-LOOKBACK+1 .. t], target = cpu[t+1 .. t+HORIZON]
T_idx = np.arange(LOOKBACK - 1, N - HORIZON)
Xw = np.stack([X_all[t - LOOKBACK + 1: t + 1] for t in T_idx])
yw = np.stack([X_all[t + 1: t + HORIZON + 1, cpu_idx] for t in T_idx])
t_end = T_idx + HORIZON                        # split on the END of the target -> no leakage
tr, va, te = t_end < i_tr, (t_end >= i_tr) & (t_end < i_va), t_end >= i_va
print("windows:", Xw.shape, "| train/val/test:", tr.sum(), va.sum(), te.sum())

lstm = models.Sequential([
    layers.Input(shape=(LOOKBACK, len(feat))),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dense(HORIZON),
])
lstm.compile(optimizer="adam", loss="mse")
hist = lstm.fit(Xw[tr], yw[tr], validation_data=(Xw[va], yw[va]), epochs=40, batch_size=128,
                callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)], verbose=2)

# ---- Evaluation vs. persistence baseline (last value repeated) ----
pred = inv_cpu(lstm.predict(Xw[te], verbose=0))
true = inv_cpu(yw[te])
naive = np.repeat(inv_cpu(Xw[te][:, -1, cpu_idx])[:, None], HORIZON, axis=1)
rmse = lambda a, b: float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
lstm_metrics = {"LSTM_MAE": float(mean_absolute_error(true.ravel(), pred.ravel())), "LSTM_RMSE": rmse(true, pred),
                "Naive_MAE": float(mean_absolute_error(true.ravel(), naive.ravel())), "Naive_RMSE": rmse(true, naive)}
print(json.dumps({k: round(v, 4) for k, v in lstm_metrics.items()}, indent=2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val")
ax[0].set_title("LSTM training loss"); ax[0].legend()
k = min(288, len(true))
ax[1].plot(true[:k, 0], label="actual (t+1)"); ax[1].plot(pred[:k, 0], label="LSTM (t+1)")
ax[1].plot(naive[:k, 0], "--", alpha=.6, label="persistence")
ax[1].set_title("CPU load forecast - first day of test set"); ax[1].legend()
plt.tight_layout(); plt.show()

# ---- Forecast for EVERY time step (needed as an input of the Random Forest) ----
T_inf = np.arange(LOOKBACK - 1, N)
X_inf = np.stack([X_all[t - LOOKBACK + 1: t + 1] for t in T_inf])
pred_vec = np.full((N, HORIZON), np.nan)
pred_vec[T_inf] = inv_cpu(lstm.predict(X_inf, verbose=0))
cpu_pred = pd.Series(pred_vec.mean(axis=1), index=data.index)

## 4. CNN — 2D thermal maps & hotspot detection

In [ ]:
n = len(sensor_cols)
G = max(3, math.ceil(math.sqrt(n)))            # grid side (sensors are laid out row by row)

def to_grid(values):                           # (t, n) -> (t, G, G); padding = row mean
    pad = G * G - n
    if pad:
        values = np.hstack([values, np.repeat(values.mean(axis=1, keepdims=True), pad, axis=1)])
    return values.reshape(-1, G, G)

grids = to_grid(data[sensor_cols].values).astype("float32")
mu, sd = grids[:i_tr].mean(), grids[:i_tr].std() + 1e-6
grids_n = (grids - mu) / sd

# Hotspot label: will the hottest sensor exceed the threshold within the next HORIZON steps?
thr = float(HOT_THRESHOLD) if HOT_THRESHOLD is not None else float(data["temp_max"].iloc[:i_tr].quantile(HOT_QUANTILE))
fut_max = data["temp_max"][::-1].rolling(HORIZON).max()[::-1].shift(-1)       # max over t+1..t+H
hot_future = (fut_max > thr).astype(float).where(fut_max.notna())
print(f"Hotspot threshold: {thr:.2f} °C | hotspot rate: {np.nanmean(hot_future.values):.1%}")

LAGS = (0, 3, 6)                               # channels = map at t, t-3, t-6 (captures the trend)
tmin = max(LAGS)
T_cnn = np.arange(tmin, N)
Xc = np.stack([np.stack([grids_n[t - l] for l in LAGS], axis=-1) for t in T_cnn]).astype("float32")
yc = hot_future.values[T_cnn]
ok = ~np.isnan(yc)
tc_end = T_cnn + HORIZON
ctr = ok & (tc_end < i_tr); cva = ok & (tc_end >= i_tr) & (tc_end < i_va); cte = ok & (tc_end >= i_va)
print("CNN input:", Xc.shape, "| train/val/test:", ctr.sum(), cva.sum(), cte.sum())

cnn = models.Sequential([
    layers.Input(shape=(G, G, len(LAGS))),
    layers.Conv2D(16, 3, padding="same", activation="relu"),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.GlobalMaxPooling2D(),               # max-pooling keeps the strongest hot spot
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])
cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])
pos = float(yc[ctr].mean())
cw = {0: 1.0, 1: float(min(5.0, (1 - pos) / max(pos, 1e-3)))}
hist_c = cnn.fit(Xc[ctr], yc[ctr], validation_data=(Xc[cva], yc[cva]), epochs=30, batch_size=128,
                 class_weight=cw, callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
                 verbose=2)

p_te = cnn.predict(Xc[cte], verbose=0).ravel()
y_te = yc[cte]
cnn_metrics = {"CNN_AUC": float(roc_auc_score(y_te, p_te)) if len(np.unique(y_te)) > 1 else float("nan"),
               "CNN_F1": float(f1_score(y_te, p_te > 0.5))}
cur_hot = (data["temp_max"].values[T_cnn][cte] > thr).astype(float)           # persistence baseline
cnn_metrics["Persistence_F1"] = float(f1_score(y_te, cur_hot))
print({k: round(v, 3) for k, v in cnn_metrics.items()})
print("Confusion matrix [[TN FP],[FN TP]]:\n", confusion_matrix(y_te, p_te > 0.5))

# ---- Hotspot probability for EVERY time step (input of the Random Forest) ----
hot_prob = pd.Series(np.nan, index=data.index)
hot_prob.iloc[T_cnn] = cnn.predict(Xc, verbose=0).ravel()

# ---- Visual: an example thermal map ----
t_show = int(np.argmax(data["temp_max"].values[i_va:])) + i_va
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
im = ax[0].imshow(grids[t_show], cmap="inferno"); ax[0].set_title(f"Thermal map at {data.index[t_show]}")
plt.colorbar(im, ax=ax[0], label="°C")
ax[1].plot(hist_c.history["loss"], label="train"); ax[1].plot(hist_c.history["val_loss"], label="val")
ax[1].set_title("CNN training loss"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Random Forest — dynamic HVAC control

> ⚠️ **Important assumption.** The public datasets contain no record of actual HVAC actions, so there is no real "setpoint" label.
> The Random Forest is trained to imitate an **oracle control policy** computed with *future* information
> (the highest safe setpoint given the true temperatures of the next 30 min). At run time it only sees information available *now*
> (LSTM forecast + CNN hotspot probability + sensors). This is a standard approach when logs of control actions are missing — say so in the report.

In [ ]:
# Oracle policy: the HIGHEST setpoint (= most efficient) that keeps the hottest sensor under the
# threshold during the next HORIZON steps, according to the simple thermal model (K_PASS).
worst_temp = np.maximum(fut_max, data["temp_max"])
sp_oracle = (BASE_SETPOINT + (thr - SAFETY_MARGIN - worst_temp) / K_PASS).clip(SP_MIN, SP_MAX)

rf_feats = ["cpu_now", "cpu_pred", "hot_prob", "temp_mean", "temp_max", "hum", "hour_sin", "hour_cos"]
F = pd.DataFrame({"cpu_now": data["cpu"], "cpu_pred": cpu_pred, "hot_prob": hot_prob,
                  "temp_mean": data["temp_mean"], "temp_max": data["temp_max"], "hum": data["hum"],
                  "hour_sin": data["hour_sin"], "hour_cos": data["hour_cos"]})
F["target"] = sp_oracle
F = F.dropna()
pos_t = data.index.get_indexer(F.index)                                        # integer position in `data`
rtr = pos_t + HORIZON < i_tr
rte = pos_t >= i_va

rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
rf.fit(F.loc[rtr, rf_feats], F.loc[rtr, "target"])
rf_pred = rf.predict(F.loc[rte, rf_feats])
rf_metrics = {"RF_MAE_degC": float(mean_absolute_error(F.loc[rte, "target"], rf_pred)),
              "RF_R2": float(r2_score(F.loc[rte, "target"], rf_pred))}
print({k: round(v, 3) for k, v in rf_metrics.items()})

imp = pd.Series(rf.feature_importances_, index=rf_feats).sort_values()
imp.plot.barh(figsize=(7, 3.5), title="Random Forest — feature importance"); plt.tight_layout(); plt.show()

def decide_setpoint(row_feats, safety=True):
    """Control law used online: RF prediction + safety layer based on CNN hotspot risk."""
    sp = float(rf.predict(pd.DataFrame([row_feats])[rf_feats])[0])
    if safety and row_feats["hot_prob"] > 0.8:
        sp = min(sp, BASE_SETPOINT - 2.0)          # high hotspot risk -> cool harder
    return float(np.clip(sp, SP_MIN, SP_MAX))

## 6. Energy, cost & carbon evaluation (test period)

In [ ]:
def cop(T):                                    # CRAC coefficient of performance vs supply temperature
    return 0.0068 * T ** 2 + 0.0008 * T + 0.458

dt_h = pd.Timedelta(STEP).total_seconds() / 3600
Te = F.loc[rte].copy()
sp_dyn = rf.predict(Te[rf_feats])
sp_dyn = np.where(Te["hot_prob"].values > 0.8, np.minimum(sp_dyn, BASE_SETPOINT - 2.0), sp_dyn)
sp_dyn = np.clip(sp_dyn, SP_MIN, SP_MAX)

p_it = N_SERVERS * (P_IDLE_KW + (P_MAX_KW - P_IDLE_KW) * Te["cpu_now"].values)          # kW
cool_base = p_it / cop(BASE_SETPOINT)
cool_dyn = p_it / cop(sp_dyn)

kwh_base, kwh_dyn = cool_base.sum() * dt_h, cool_dyn.sum() * dt_h
saved = kwh_base - kwh_dyn
pue_base = float(((p_it + cool_base) / p_it).mean())
pue_dyn = float(((p_it + cool_dyn) / p_it).mean())

# Thermal safety (simple model; the observed data is assumed to be the baseline policy)
tmax_dyn = Te["temp_max"].values + K_PASS * (sp_dyn - BASE_SETPOINT)
viol_base = float((Te["temp_max"].values > thr).mean())
viol_dyn = float((tmax_dyn > thr).mean())

results = {
    "cooling_kWh_baseline": round(kwh_base, 1), "cooling_kWh_AI": round(kwh_dyn, 1),
    "energy_saved_kWh": round(saved, 1), "energy_saved_pct": round(100 * saved / kwh_base, 2),
    "CO2_avoided_kg": round(saved * CO2_KG_PER_KWH, 1), "cost_saved": round(saved * PRICE_PER_KWH, 2),
    "PUE_baseline": round(pue_base, 3), "PUE_AI": round(pue_dyn, 3),
    "time_above_hot_threshold_baseline_pct": round(100 * viol_base, 2),
    "time_above_hot_threshold_AI_pct": round(100 * viol_dyn, 2),
    "test_days": round(len(Te) * dt_h / 24, 1),
}
print(json.dumps(results, indent=2))

fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
k = min(288 * 3, len(Te)); ix = Te.index[:k]
ax[0].plot(ix, Te["cpu_now"].values[:k]); ax[0].set_ylabel("CPU load")
ax[1].plot(ix, sp_dyn[:k], label="AI setpoint"); ax[1].axhline(BASE_SETPOINT, color="r", ls="--", label="fixed baseline")
ax[1].set_ylabel("Setpoint (°C)"); ax[1].legend()
ax[2].plot(ix, cool_base[:k], "r--", label="baseline"); ax[2].plot(ix, cool_dyn[:k], label="AI")
ax[2].set_ylabel("Cooling power (kW)"); ax[2].legend()
plt.suptitle("Dynamic cooling vs. fixed setpoint (first 3 days of the test set)"); plt.tight_layout(); plt.show()

## 7. Demo — one decision step & saving the deliverables

In [ ]:
def demo(position):
    """Show what the system does at one time step: forecast -> thermal map -> HVAC decision."""
    ts = F.index[position]; row = F.loc[ts, rf_feats].to_dict()
    t = data.index.get_loc(ts)
    sp = decide_setpoint(row)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(range(-LOOKBACK + 1, 1), data["cpu"].values[t - LOOKBACK + 1: t + 1], label="history")
    ax[0].plot(range(1, HORIZON + 1), pred_vec[t], "o-", label="LSTM forecast")
    ax[0].set_title("CPU load"); ax[0].set_xlabel("steps"); ax[0].legend()
    im = ax[1].imshow(grids[t], cmap="inferno"); plt.colorbar(im, ax=ax[1], label="°C")
    ax[1].set_title(f"Thermal map | hotspot risk = {row['hot_prob']:.0%}")
    ax[2].axis("off")
    ax[2].text(0, .6, f"HVAC supply setpoint\n{sp:.1f} °C", fontsize=22, weight="bold")
    ax[2].text(0, .25, f"(fixed baseline: {BASE_SETPOINT:.0f} °C)\n{ts}", fontsize=11)
    plt.tight_layout(); plt.show()

demo(int(np.where(rte)[0][10]))                          # a calm moment
demo(int(np.where(rte)[0][np.argmax(Te["hot_prob"].values)]))   # highest hotspot risk

# ---- Save everything ----
lstm.save(f"{OUT}/lstm_cpu_forecast.keras")
cnn.save(f"{OUT}/cnn_hotspot.keras")
import joblib; joblib.dump(rf, f"{OUT}/rf_hvac_controller.joblib", compress=3)
all_metrics = {**lstm_metrics, **cnn_metrics, **rf_metrics, **results, "synthetic_data": USING_SYNTHETIC}
json.dump(all_metrics, open(f"{OUT}/metrics.json", "w"), indent=2)
pd.DataFrame({"cpu_load": Te["cpu_now"].values, "setpoint_AI": sp_dyn, "cooling_kW_baseline": cool_base,
              "cooling_kW_AI": cool_dyn}, index=Te.index).to_csv(f"{OUT}/test_control_log.csv")
print("Saved to", OUT, "->", os.listdir(OUT))

## 8. Notes for the technical report
- **Assumptions to state:** time alignment of the two datasets, sensor-to-grid layout, oracle setpoint policy, energy model (server power, CRAC COP curve, K_PASS, emission factor, price).
- **Honest limits:** RF/CNN are partly evaluated in-sample for the RF training period; savings come from a model, not from a real data center; with real data, validate on a held-out period and with physical measurements.
- **Possible extensions:** multi-step LSTM with attention, real rack coordinates for the CNN, reinforcement learning for the controller, a Streamlit/Gradio front-end for the demo.